# Spambase : détection de spam

**Mini-projet Machine Learning — TAF MCE (IMT Atlantique)**

Ce notebook applique le workflow défini dans `ml_workflow.py` au jeu **Spambase** (UCI) :
1. import des données ; 2. nettoyage et prétraitement ; 3. séparation train / test ;
4. entraînement avec sélection de variables ; 5. validation.

Toutes les fonctions appelées ici sont définies dans `ml_workflow.py` (préfixe `mw.`).

## 0. Le problème et le choix de la métrique

On veut décider si un e-mail est un spam (classe 1) à partir de 57 statistiques sur son texte.
Les deux erreurs n'ont pas le même coût. La documentation du jeu le dit explicitement :
*« False positives (marking good mail as spam) are very undesirable »*. Un faux positif fait
disparaître un vrai e-mail dans le dossier spam ; un faux négatif laisse passer une publicité.

C'est exactement le cadre du **test de Neyman-Pearson** vu en cours : on fixe un budget d'erreur
de type I (faux positif) $\alpha$, et on maximise la puissance (le rappel) sous cette contrainte.
On retient donc :

| Rôle | Métrique |
|---|---|
| Comparer les modèles indépendamment du seuil | **AUC-ROC** |
| Critère métier (Neyman-Pearson) | **rappel à taux de faux positifs ≤ 1 %** (`tpr_fpr1`) |
| Lisibilité (le jeu est presque équilibré) | accuracy, précision, F1 |

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ml_workflow as mw

SEED = 42
pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 90

: 

## 1. Import et contrôle des données

In [ ]:
df_raw = mw.load_spambase("data")
print(df_raw.shape)
mw.describe_dataset(df_raw, "is_spam")

- **Pas de valeur manquante** : l'étape d'imputation reste dans le pipeline (elle sera testée
  à la section 4.2 en retirant artificiellement des valeurs).
- Le sujet annonce « 5000 entrées, 50 features » : les vrais chiffres sont **4 601 lignes et 57 variables**.
- **391 lignes sont des doublons exacts.** Ici, une ligne dupliquée est très probablement le
  même e-mail (57 fréquences continues identiques au centième près). Si on les garde, un même
  e-mail peut tomber à la fois dans le train et dans le test, et le score de test devient optimiste.
  **On les supprime avant la séparation** : c'est la seule opération autorisée avant le split,
  car elle n'apprend rien des données.
- Après suppression, 3 profils restent présents avec les deux labels : ils sont gardés (bruit d'étiquetage irréductible).

In [ ]:
df = mw.remove_duplicates(df_raw)
X, y = mw.split_features_target(df, "is_spam")
print(f"Après suppression des doublons : {X.shape[0]} e-mails, {X.shape[1]} variables, {y.mean():.1%} de spams")
mw.plot_class_balance(y, labels=("non-spam", "spam"));

Le jeu est **à peu près équilibré** (40 % de spams). Le classifieur trivial « toujours non-spam »
obtient 60 % d'accuracy : c'est la référence à battre.

### Nature des variables

In [ ]:
desc = pd.DataFrame({
    "part_de_zeros": (X == 0).mean(),
    "moyenne": X.mean(),
    "max": X.max(),
    "asymetrie": X.skew(),
})
print("Part moyenne de zéros :", f"{(X == 0).values.mean():.1%}")
desc.sort_values("max", ascending=False).head(8)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 5))
for j, col in enumerate(["word_freq_free", "char_freq_!", "capital_run_length_average"]):
    for c, lab in [(0, "non-spam"), (1, "spam")]:
        v = X.loc[(y == c) & (X[col] > 0), col]          # valeurs non nulles seulement
        axes[0, j].hist(v, bins=50, alpha=0.6, label=lab)
        axes[1, j].hist(np.log1p(v), bins=50, alpha=0.6, label=lab)
    axes[0, j].set_title(f"{col} (valeurs > 0)")
    axes[1, j].set_title(f"log(1 + {col})")
axes[0, 0].legend()
plt.tight_layout()

Trois constats qui guident le prétraitement :

1. **Les échelles sont sans commune mesure** : les fréquences sont des pourcentages (≤ 100),
   alors que `capital_run_length_total` monte à 15 841. Sans centrage-réduction, toute distance
   (kNN, SVM) ou toute pénalité (régression logistique régularisée) serait dominée par cette variable.
2. **Les variables sont très creuses** (environ 3/4 de zéros) et **très asymétriques** (queues lourdes).
   La transformation $x \mapsto \log(1+x)$ écrase ces queues (ligne du bas). On s'attend à ce qu'elle
   aide les modèles linéaires et à distance, et qu'elle soit **sans effet sur les arbres**, qui ne
   dépendent que de l'ordre des valeurs (une transformation croissante ne change aucun seuil).
3. L'**hypothèse gaussienne** des modèles génératifs du cours (LDA, QDA) est clairement fausse sur
   ces données brutes (masse en 0 + queue lourde). On mesurera ce que cela coûte.

## 2. Séparation train / test

La séparation est faite **avant** tout apprentissage (centrage, ACP, sélection), et elle est
**stratifiée** pour garder 40 % de spams dans les deux parties. Le test (20 %) n'est utilisé
qu'**une seule fois**, à la toute fin (section 7). Tous les choix intermédiaires (prétraitement,
modèle, hyperparamètres, variables, seuil) sont faits par **validation croisée 5 plis stratifiée** sur le train.

In [ ]:
X_train, X_test, y_train, y_test = mw.split_data(X, y, test_size=0.2, seed=SEED)
cv = mw.make_cv(5, SEED)
print(f"train : {X_train.shape}, {y_train.mean():.3f} de spams | test : {X_test.shape}, {y_test.mean():.3f} de spams")

SCORING = {
    "auc_roc": "roc_auc",
    "tpr_fpr1": mw.tpr_at_fpr_scorer(0.01),
    "accuracy": "accuracy",
    "precision": "precision",
    "f1": "f1",
}

## 3. Prétraitement

### 3.1 Centrage-réduction codé à la main

`mw.StandardScalerManual` calcule $\mu$ et $\sigma$ **sur le train seulement** et les applique au
test. On vérifie qu'il donne exactement le même résultat que `sklearn.preprocessing.StandardScaler`
(c'est aussi l'objet d'un test unitaire).

In [ ]:
from sklearn.preprocessing import StandardScaler
ours = mw.StandardScalerManual().fit(X_train)
print("Identique à sklearn :", np.allclose(ours.transform(X_test), StandardScaler().fit(X_train).transform(X_test)))
Z_test = ours.transform(X_test)
print(f"Sur le TEST : moyennes dans [{Z_test.mean(0).min():.2f}, {Z_test.mean(0).max():.2f}], "
      f"écarts-types dans [{Z_test.std(0).min():.2f}, {Z_test.std(0).max():.2f}]")

Sur le test, les moyennes ne sont pas exactement nulles ni les écarts-types exactement 1 : c'est
normal et c'est le signe qu'on n'a **pas** utilisé le test pour apprendre $\mu$ et $\sigma$.
Les écarts-types très grands sur certaines variables viennent des queues lourdes (une variable
quasi nulle sur le train peut prendre de grandes valeurs sur quelques e-mails du test).

### 3.2 Valeurs manquantes : moyenne ou médiane ?

Le jeu est complet, mais la consigne demande de gérer les valeurs manquantes. On retire au hasard
10 % puis 30 % des valeurs (au train et au test), on impute par la moyenne ou par la médiane
**apprise sur le train**, et on mesure l'AUC-ROC sur le test pour deux modèles.
(Cette expérience regarde le test, mais elle ne sert à aucun choix du modèle final : elle sert
seulement à illustrer l'imputation.)

In [ ]:
for name, model in [("Régression logistique", mw.LogisticRegressionGD()),
                    ("Gradient Boosting", mw.get_models(SEED)["Gradient Boosting (hist.)"])]:
    res = mw.missing_values_experiment(
        lambda s: mw.build_pipeline(model, impute=s, log_transform=True),
        X_train, y_train, X_test, y_test, rates=(0.0, 0.1, 0.3), seed=SEED)
    print(name); display(res)

- Sans valeur manquante, les deux stratégies sont identiques (rien à imputer).
- Avec 30 % de valeurs manquantes, la **médiane est meilleure** pour les deux modèles (régression
  logistique : 0,971 contre 0,963 d'AUC ; boosting : 0,978 contre 0,975). À 10 %, elle est meilleure
  pour la régression logistique et équivalente pour le boosting.
- **Pourquoi** : pour la plupart des variables, 3/4 des valeurs sont nulles et la distribution a une
  queue lourde. La médiane vaut donc 0 (la valeur la plus probable pour un e-mail), alors que la moyenne
  est tirée vers le haut par quelques valeurs extrêmes : imputer par la moyenne revient à inventer des
  occurrences de mots qui n'existent pas dans un e-mail typique.
- Même avec 30 % de trous, la perte reste d'environ 0,01 d'AUC : les variables sont redondantes
  (plusieurs mots « commerciaux » portent la même information).

**Choix retenu : imputation par la médiane** (inactive ici puisque le jeu est complet).

## 4. ACP (codée à la main)

`mw.PCAManual` diagonalise la matrice de covariance $C = \frac{1}{n-1}X_c^\top X_c = V\Lambda V^\top$
des données centrées-réduites (après $\log(1+x)$) et projette sur les $k$ premiers vecteurs propres.
Les valeurs propres $\lambda_i$ sont les variances le long des axes.

In [ ]:
from sklearn.decomposition import PCA
prep = mw.build_pipeline(None, log_transform=True)      # imputation -> log1p -> centrage-réduction
Z_train = prep.fit_transform(X_train)
pca = mw.PCAManual(n_components=0.95).fit(Z_train)
ref = PCA(n_components=pca.n_components_).fit(Z_train)
print("Variances expliquées identiques à sklearn :", np.allclose(pca.explained_variance_, ref.explained_variance_))
print("Axes identiques au signe près :", np.allclose(np.abs(pca.components_), np.abs(ref.components_)))
print("Signes des 10 premiers axes, nous vs sklearn :",
      np.sign((pca.components_[:10] * ref.components_[:10]).sum(1)).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
mw.plot_pca_variance(pca, 0.95, ax=axes[0])
mw.plot_pca_2d(pca.transform(Z_train), y_train, labels=("non-spam", "spam"), ax=axes[1])
plt.tight_layout()

- Notre ACP donne exactement les mêmes variances et les mêmes axes que scikit-learn. Les signes
  coïncident parce qu'on a adopté la même convention (la plus grande composante de chaque axe est
  rendue positive) ; sans convention, un axe et son opposé sont des solutions également valables
  (si $v$ est vecteur propre, $-v$ aussi).
- **Il faut 47 axes sur 57 pour garder 95 % de la variance**, et le premier n'en explique que 13 %.
  La variance est très étalée : les variables sont peu redondantes **linéairement**, donc l'ACP
  comprime mal ce jeu.
- La projection sur PC1–PC2 montre une séparation partielle (les spams occupent une « branche »),
  avec un fort recouvrement. C'est attendu : l'ACP ne regarde pas le label ; les directions de grande
  variance ne sont pas forcément celles qui séparent les classes. L'effet de l'ACP sur la
  classification est mesuré à la section 6.4.